# Compute outputs of OSMCLIP model on images and filters.

This data is used to quantify the effects of applying aesthetic filters to images.

In [ ]:
from PIL import Image
from models.huggingface import Geolocalizer
import torch
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from glob import glob
import math
import os

## Load model

In [ ]:
!git clone https://github.com/gastruc/osv5m

In [ ]:
%cd /kaggle/working/osv5m

In [ ]:
%pip install -r requirements.txt

## Setup

In [ ]:
os.makedirs("./csv_files", exist_ok=True)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
geoloc = Geolocalizer.from_pretrained("osv5m/baseline").to(device)
geoloc

**Note**: Update this path accordingly

In [ ]:
base_dir = "path_to_filtered_dataset"  # Update this path accordingly

In [ ]:
filter_names = os.listdir(base_dir)
print(filter_names)

## Run the model on different images with filters applied

In [ ]:
for filter_name in filter_names:
    print(f"Processing {filter_name}")

    # List all images
    all_images = glob(f"{os.path.join(base_dir, filter_name)}/*.jpg")

    # Store image ID and predicted latitude and longitude
    all_image_ids = []
    all_coords = []
    for i in tqdm(range(len(all_images))):
        image_path = all_images[i]
        score_key = Path(image_path).stem
        with torch.no_grad():
            img = Image.open(image_path).convert("RGB")
            x = geoloc.transform(img).unsqueeze(0).to(device)
            top_pred_gps = geoloc(x)[0]
            lat_rad = float(top_pred_gps[0].detach().cpu())
            lon_rad = float(top_pred_gps[1].detach().cpu())
            # Convert radians to degrees
            lat_deg = (lat_rad * 180) / math.pi
            lon_deg = (lon_rad * 180) / math.pi

            predicted_loc = (str(lat_deg), str(lon_deg))

            all_image_ids.append(score_key)
            all_coords.append(predicted_loc)
        # Clear cache
        if i % 1000 == 0:
            torch.cuda.empty_cache()
            df = pd.DataFrame({"image_id": all_image_ids, "comp_coords": all_coords})
            df.to_csv(f"./csv_files/osmclip_{filter_name}.csv", index=False)


    # Save results to CSV file
    df = pd.DataFrame({"image_id": all_image_ids, "comp_coords": all_coords})
    df.to_csv(f"./csv_files/osmclip_{filter_name}.csv", index=False)